## 1. Load the coding-variant dataset
Loads `wanglab/variant_effect_coding` from Hugging Face and inspects its structure (columns, row counts, sample answers).

In [1]:
from datasets import load_dataset

ds_coding = load_dataset("wanglab/variant_effect_coding")
print(ds_coding)

test_coding_df = ds_coding['test'].to_pandas()
print("\nColumns:", test_coding_df.columns.tolist())
print("\nTotal test rows:", len(test_coding_df))

print("\n--- Sample answer values ---")
print(test_coding_df['answer'].head(5).tolist())

DatasetDict({
    train: Dataset({
        features: ['ID', 'question', 'answer', 'reference_sequence', 'variant_sequence'],
        num_rows: 48850
    })
    test: Dataset({
        features: ['ID', 'question', 'answer', 'reference_sequence', 'variant_sequence'],
        num_rows: 1233
    })
})

Columns: ['ID', 'question', 'answer', 'reference_sequence', 'variant_sequence']

Total test rows: 1233

--- Sample answer values ---
['Pathogenic; Neuronal ceroid lipofuscinosis', 'Pathogenic; Neuronal ceroid lipofuscinosis 8', 'Pathogenic; Neuronal ceroid lipofuscinosis 8', 'Pathogenic; Neuronal ceroid lipofuscinosis 8 northern epilepsy variant', 'Pathogenic; Neuronal ceroid lipofuscinosis 8']


## 2. Derive binary true labels
Extracts a clean `Pathogenic`/`Benign` label from the `answer` column.

In [2]:
test_coding_df['true_label'] = test_coding_df['answer'].str.split(';').str[0].str.strip()

print(test_coding_df['true_label'].value_counts())
print("\nTotal:", len(test_coding_df))

true_label
Pathogenic    676
Benign        557
Name: count, dtype: int64

Total: 1233


## 3. Sample 200 stratified test rows
Draws a stratified 200-row sample from the test set to use for evaluation.

In [3]:
from sklearn.model_selection import train_test_split

sample_coding_200_df, _ = train_test_split(
    test_coding_df,
    train_size=200,
    stratify=test_coding_df['true_label'],
    random_state=42
)
sample_coding_200_df = sample_coding_200_df.reset_index(drop=True)

print(sample_coding_200_df['true_label'].value_counts())
print("Sample size:", len(sample_coding_200_df))


true_label
Pathogenic    110
Benign         90
Name: count, dtype: int64
Sample size: 200


## 4. Define the qwen3:1.7b classifier
`classify_variant()` sends a prompt to the local `qwen3:1.7b` model (via Ollama) and returns its reasoning/content.

In [4]:
def classify_variant(question_text, model_name="qwen3:1.7b", max_retries=2):
    """Original settings matching the VE-Non-SNV 1.7b run: max_tokens=1500, default context window."""
    prompt = f"""{question_text}

After your analysis, you MUST end with exactly this line and nothing else after it:
Final Answer: [Pathogenic|Benign]"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                max_tokens=1500
            )
            message = response.choices[0].message
            return {
                "reasoning": getattr(message, "reasoning", None),
                "content": message.content,
                "error": None
            }
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")

    return {"reasoning": None, "content": None, "error": "Failed after retries"}

## 5. Run batch inference (qwen3:1.7b, attempt 1)
Loops over the 200-row sample, calling `classify_variant`, and appends results to `results_ve_coding_200_1p7b.jsonl` (resumable by row index).

In [5]:
import json
import os

output_file = "results_ve_coding_200_1p7b.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in sample_coding_200_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 10 == 0:
        print(f"Processing {idx+1}/{len(sample_coding_200_df)}...")

    result = classify_variant(row['question'], model_name="qwen3:1.7b")

    record = {
        "row_index": idx,
        "true_label": row['true_label'],
        "reasoning": result['reasoning'],
        "content": result['content'],
        "error": result['error']
    }

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done.")

Already completed: 1 rows
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Processing 11/200...
Attempt 1 failed: name 'client' is not defined
Attempt 2 failed: name 'client' is not defined
Attempt 1 fai

## 6. Inspect results — attempt 1
Loads the results file back and checks the first/last records.

In [6]:
import json

with open("results_ve_coding_200_1p7b.jsonl", 'r') as f:
    lines = [json.loads(line) for line in f]

print("Total lines in file:", len(lines))
print("\n--- First result ---")
print("Content:", lines[0]['content'])
print("Reasoning length:", len(lines[0]['reasoning']) if lines[0]['reasoning'] else 0)
print("\n--- Last result ---")
print("Content:", lines[-1]['content'])
print("Reasoning length:", len(lines[-1]['reasoning']) if lines[-1]['reasoning'] else 0)

empty_count = sum(1 for l in lines if not l['content'] and not l['reasoning'])
print(f"\nRows with both content AND reasoning empty: {empty_count}")

Total lines in file: 200

--- First result ---
Content: Final Answer: [Pathogenic]
Reasoning length: 3971

--- Last result ---
Content: None
Reasoning length: 0

Rows with both content AND reasoning empty: 199


## 7. Check the error field
Confirms every row failed with `"Failed after retries"` — all content/reasoning are empty.

In [7]:
print("Error field for row 0:", lines[0]['error'])
print("Error field for row 50:", lines[50]['error'])
print("Error field for row 199:", lines[199]['error'])

Error field for row 0: None
Error field for row 50: Failed after retries
Error field for row 199: Failed after retries


## 8. Debug the failure
Checks Ollama connectivity and reproduces the root cause: `client` was never defined before the batch loop ran.

In [8]:
# Check Ollama connectivity first
try:
    models = client.models.list()
    print("Connected. Available models:")
    for m in models.data:
        print(" -", m.id)
except Exception as e:
    print("Connection failed:", e)

# Now try a single classification call and print the FULL error
import traceback

try:
    response = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=[{"role": "user", "content": sample_coding_200_df.iloc[0]['question']}],
        temperature=0.6,
        max_tokens=1500
    )
    print("SUCCESS:", response.choices[0].message.content)
except Exception as e:
    print("FAILED WITH:")
    traceback.print_exc()

Connection failed: name 'client' is not defined
FAILED WITH:


Traceback (most recent call last):
  File "C:\Users\Araf\AppData\Local\Temp\ipykernel_22452\2584143309.py", line 14, in <module>
    response = client.chat.completions.create(
NameError: name 'client' is not defined


## 9. Fix: define the OpenAI/Ollama client
Creates the `client` pointed at the local Ollama server, redefines `classify_variant`, and adds `parse_prediction`/`parse_prediction_v3` helpers to extract the final label from model output.

In [9]:
from openai import OpenAI
import re

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

def parse_prediction(content):
    if not content:
        return "Uncertain"
    match = re.search(r"Final Answer:\s*\[?(Pathogenic|Benign)", content, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Uncertain"

def parse_prediction_v3(content, reasoning):
    result = parse_prediction(content)
    if result != "Uncertain":
        return result
    if not reasoning:
        return "Uncertain"
    matches = re.findall(r"Final Answer:\s*\[?(Pathogenic|Benign)\]?", reasoning, re.IGNORECASE)
    if matches:
        return matches[-1].capitalize()
    return "Uncertain"

def classify_variant(question_text, model_name="qwen3:1.7b", max_retries=2):
    prompt = f"""{question_text}

After your analysis, you MUST end with exactly this line and nothing else after it:
Final Answer: [Pathogenic|Benign]"""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                max_tokens=1500
            )
            message = response.choices[0].message
            return {
                "reasoning": getattr(message, "reasoning", None),
                "content": message.content,
                "error": None
            }
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
    return {"reasoning": None, "content": None, "error": "Failed after retries"}

# Verify it works now
models = client.models.list()
for m in models.data:
    print(m.id)

qwen3:4b
qwen3:1.7b


## 10. Reset the broken results file
Deletes the corrupted `results_ve_coding_200_1p7b.jsonl` so the batch loop can be re-run cleanly.

In [10]:
import os
os.remove("results_ve_coding_200_1p7b.jsonl")
print("Deleted broken results file. Ready to re-run the batch loop.")

Deleted broken results file. Ready to re-run the batch loop.


## 11. Run batch inference (qwen3:1.7b, attempt 2)
Re-runs the classification loop now that `client` is defined; completes all 200 rows successfully.

In [11]:
import json
import os

output_file = "results_ve_coding_200_1p7b.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in sample_coding_200_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 10 == 0:
        print(f"Processing {idx+1}/{len(sample_coding_200_df)}...")

    result = classify_variant(row['question'], model_name="qwen3:1.7b")

    record = {
        "row_index": idx,
        "true_label": row['true_label'],
        "reasoning": result['reasoning'],
        "content": result['content'],
        "error": result['error']
    }

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done.")

Already completed: 0 rows
Processing 1/200...
Processing 11/200...
Processing 21/200...
Processing 31/200...
Processing 41/200...
Processing 51/200...
Processing 61/200...
Processing 71/200...
Processing 81/200...
Processing 91/200...
Processing 101/200...
Processing 111/200...
Processing 121/200...
Processing 131/200...
Processing 141/200...
Processing 151/200...
Processing 161/200...
Processing 171/200...
Processing 181/200...
Processing 191/200...
Done.


## 12. Inspect results — attempt 2
Confirms no rows have empty content/reasoning this time.

In [12]:
import json

with open("results_ve_coding_200_1p7b.jsonl", 'r') as f:
    lines = [json.loads(line) for line in f]

print("Total lines in file:", len(lines))
print("\n--- First result ---")
print("Content:", lines[0]['content'])
print("Reasoning length:", len(lines[0]['reasoning']) if lines[0]['reasoning'] else 0)

empty_count = sum(1 for l in lines if not l['content'] and not l['reasoning'])
print(f"\nRows with both content AND reasoning empty: {empty_count}")

Total lines in file: 200

--- First result ---
Content: Final Answer: [Pathogenic]
Reasoning length: 4106

Rows with both content AND reasoning empty: 0


## 13. Check for unfilled answer templates
Counts rows where the model literally echoed the `[Pathogenic|Benign]` template instead of picking one.

In [13]:
literal_copy_count = sum(1 for l in lines if l['content'] and '[Pathogenic|Benign]' in l['content'])
print(f"Rows with literal unfilled template: {literal_copy_count} / {len(lines)}")

# Show a few examples
for l in lines[:5]:
    print(repr(l['content']))

Rows with literal unfilled template: 12 / 200
'Final Answer: [Pathogenic]'
'Final Answer: Pathogenic / Myotonic Dystrophy Type 1'
'Final Answer: Pathogenic / PXD1'
'Final Answer: [Pathogenic]'
'Final Answer: Benign'


## 14. Improve the label parser
Tightens the regex so a literal `[Pathogenic|Benign]` copy isn't mistaken for a real prediction, with a reasoning-text fallback.

In [14]:
import re

def parse_prediction(content):
    if not content:
        return "Uncertain"
    # (?!\|) rejects a match if the word is immediately followed by '|' — 
    # that pattern only occurs when the model copied "[Pathogenic|Benign]" literally
    match = re.search(r"Final Answer:\s*\[?(Pathogenic|Benign)(?!\|)", content, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Uncertain"

def parse_prediction_v3(content, reasoning):
    result = parse_prediction(content)
    if result != "Uncertain":
        return result
    if not reasoning:
        return "Uncertain"
    matches = re.findall(r"Final Answer:\s*\[?(Pathogenic|Benign)(?!\|)\]?", reasoning, re.IGNORECASE)
    if matches:
        return matches[-1].capitalize()
    return "Uncertain"

## 15. Build the results dataframe (1.7b)
Applies the improved parser to get `predicted_label` for every row and verifies the literal-copy rows now resolve via the reasoning fallback.

In [15]:
import pandas as pd

results_coding_1p7b_df = pd.DataFrame(lines)
results_coding_1p7b_df['predicted_label'] = results_coding_1p7b_df.apply(
    lambda row: parse_prediction_v3(row['content'], row['reasoning']), axis=1
)

print(results_coding_1p7b_df['predicted_label'].value_counts())

# Double check: none of the 12 literal-copy rows should now show as a false "Pathogenic" from content alone
literal_rows = results_coding_1p7b_df[results_coding_1p7b_df['content'].str.contains(r'\[Pathogenic\|Benign\]', na=False, regex=True)]
print("\nLiteral-copy rows, now resolved via reasoning fallback:")
print(literal_rows[['true_label', 'content', 'predicted_label']])

predicted_label
Pathogenic    109
Benign         67
Uncertain      24
Name: count, dtype: int64

Literal-copy rows, now resolved via reasoning fallback:
     true_label                                            content  \
54   Pathogenic                  Final Answer: [Pathogenic|Benign]   
72   Pathogenic                  Final Answer: [Pathogenic|Benign]   
79   Pathogenic                  Final Answer: [Pathogenic|Benign]   
114      Benign         Final Answer: Benign / [Pathogenic|Benign]   
125      Benign  The variant in question is likely a pathogenic...   
129  Pathogenic                  Final Answer: [Pathogenic|Benign]   
132  Pathogenic  The variant in the HGSNAT gene, which is invol...   
142      Benign                  Final Answer: [Pathogenic|Benign]   
161  Pathogenic                  Final Answer: [Pathogenic|Benign]   
180  Pathogenic                  Final Answer: [Pathogenic|Benign]   
183      Benign         Final Answer: Benign / [Pathogenic|Benign]   
190  Pa

## 16. Inspect remaining "Uncertain" rows
Reviews the raw reasoning text for rows the parser still couldn't resolve.

In [16]:
uncertain_now = results_coding_1p7b_df[results_coding_1p7b_df['predicted_label'] == 'Uncertain']
print(f"Total uncertain: {len(uncertain_now)}")

# Look at the full reasoning of a few, not just the tail
for i, row in uncertain_now.head(3).iterrows():
    print(f"--- Row {i} (true: {row['true_label']}) ---")
    print(row['reasoning'][-500:] if row['reasoning'] else "No reasoning")
    print()

Total uncertain: 24
--- Row 11 (true: Benign) ---
e safest is to say Benign, but that's not correct. However, the user might expect a different answer. 

Alternatively, if the mutation is in a gene that's associated with a syndrome, like if it's a deletion or duplication, then it's pathogenic. But without knowing, it's impossible. 

Since the user provided the context that the variant is on chromosome 8, but not the specific gene, the answer cannot be determined. However, the system requires an answer. Therefore, maybe the answer is Benign, but

--- Row 15 (true: Benign) ---
determine. But since the system requires an answer, perhaps the answer is "Benign" as a default. But this is not accurate. 

Wait, maybe the answer is "Pathogenic" because chromosome 8 is associated with several diseases, so a mutation there could be pathogenic. But again, without specifics, it's a guess. 

Given the ambiguity, the answer is that it's not possible to determine without more information. But since th

## 17. Evaluate qwen3:1.7b performance
Computes accuracy, precision, recall, and F1 (macro and per-class) against the true labels.

In [17]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, f1_score, precision_score, recall_score

y_true_coding_1p7b = results_coding_1p7b_df['true_label']
y_pred_coding_1p7b = results_coding_1p7b_df['predicted_label']

accuracy_c1 = accuracy_score(y_true_coding_1p7b, y_pred_coding_1p7b)
precision_c1, recall_c1, f1_c1, _ = precision_recall_fscore_support(
    y_true_coding_1p7b, y_pred_coding_1p7b, labels=['Pathogenic', 'Benign'], average='macro', zero_division=0
)

f1_scores_c1 = f1_score(y_true_coding_1p7b, y_pred_coding_1p7b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
precision_scores_c1 = precision_score(y_true_coding_1p7b, y_pred_coding_1p7b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
recall_scores_c1 = recall_score(y_true_coding_1p7b, y_pred_coding_1p7b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)

print(f"Accuracy:  {accuracy_c1*100:.2f}%")
print(f"Macro Precision: {precision_c1*100:.2f}%")
print(f"Macro Recall:    {recall_c1*100:.2f}%")
print(f"Macro F1-Score:  {f1_c1*100:.2f}%")

print(f"\nPathogenic-class F1:        {f1_scores_c1[0]*100:.2f}%")
print(f"Pathogenic-class Precision: {precision_scores_c1[0]*100:.2f}%")
print(f"Pathogenic-class Recall:    {recall_scores_c1[0]*100:.2f}%")

print("\n--- Full classification report ---")
print(classification_report(y_true_coding_1p7b, y_pred_coding_1p7b, labels=['Pathogenic', 'Benign'], zero_division=0))

Accuracy:  70.00%
Macro Precision: 79.75%
Macro Recall:    69.09%
Macro F1-Score:  73.66%

Pathogenic-class F1:        78.54%
Pathogenic-class Precision: 78.90%
Pathogenic-class Recall:    78.18%

--- Full classification report ---
              precision    recall  f1-score   support

  Pathogenic       0.79      0.78      0.79       110
      Benign       0.81      0.60      0.69        90

   micro avg       0.80      0.70      0.74       200
   macro avg       0.80      0.69      0.74       200
weighted avg       0.80      0.70      0.74       200



## 18. Define the qwen3:4b classifier
`classify_variant_v2()` — same idea as before but with a larger `max_tokens` and explicit context window sized for the bigger model.

In [18]:
def classify_variant_v2(question_text, model_name="qwen3:4b", max_retries=2, max_tokens=4000):
    """4b needs more room: higher max_tokens + explicit context window to prevent silent truncation."""
    prompt = f"""{question_text}

After your analysis, you MUST end with exactly this line and nothing else after it:
Final Answer: [Pathogenic|Benign]"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                max_tokens=max_tokens,
                extra_body={"options": {"num_ctx": 8192}}
            )
            message = response.choices[0].message
            return {
                "reasoning": getattr(message, "reasoning", None),
                "content": message.content,
                "error": None
            }
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")

    return {"reasoning": None, "content": None, "error": "Failed after retries"}

## 19. Run batch inference (qwen3:4b, attempt 1)
Runs the classification loop for `qwen3:4b` over the same 200-row sample.

In [19]:
import json
import os

output_file = "results_ve_coding_200_4b.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in sample_coding_200_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 10 == 0:
        print(f"Processing {idx+1}/{len(sample_coding_200_df)}...")

    result = classify_variant_v2(row['question'], model_name="qwen3:4b")

    record = {
        "row_index": idx,
        "true_label": row['true_label'],
        "reasoning": result['reasoning'],
        "content": result['content'],
        "error": result['error']
    }

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done.")

Already completed: 200 rows
Done.


## 20. Resume batch inference (qwen3:4b)
Re-runs the loop to pick up any rows not completed in the first pass; finishes all 200 rows.

In [20]:
import json
import os

output_file = "results_ve_coding_200_4b.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in sample_coding_200_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 10 == 0:
        print(f"Processing {idx+1}/{len(sample_coding_200_df)}...")

    result = classify_variant_v2(row['question'], model_name="qwen3:4b")

    record = {"row_index": idx, "true_label": row['true_label'], "reasoning": result['reasoning'], "content": result['content'], "error": result['error']}

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done.")

Already completed: 200 rows
Done.


## 21. Inspect results (qwen3:4b)
Checks the 4b results file for empty content/reasoning or recorded errors.

In [21]:
import json

with open("results_ve_coding_200_4b.jsonl", 'r') as f:
    lines_4b = [json.loads(line) for line in f]

print("Total lines in file:", len(lines_4b))

empty_count = sum(1 for l in lines_4b if not l['content'] and not l['reasoning'])
print(f"Rows with both content AND reasoning empty: {empty_count}")

error_count = sum(1 for l in lines_4b if l['error'])
print(f"Rows with an error recorded: {error_count}")

print("\n--- First result ---")
print("Content:", lines_4b[0]['content'])
print("Reasoning length:", len(lines_4b[0]['reasoning']) if lines_4b[0]['reasoning'] else 0)

Total lines in file: 200
Rows with both content AND reasoning empty: 0
Rows with an error recorded: 0

--- First result ---
Content: The gene CHD7 (chromodomain helicase DNA binding protein 7) is located on chromosome 8 (specifically 8q24.3). Pathogenic variants in CHD7 are well-established to cause CHD7 syndrome, an autosomal dominant disorder characterized by a range of clinical features including craniofacial abnormalities (e.g., hypertelorism, microcephaly), cardiac defects, renal anomalies, and intellectual disability. This syndrome is also known as CHD7-related disorder or CHD7 syndrome. 

In clinical genetics, variants in CHD7 that are associated with CHD7 syndrome are classified as pathogenic due to strong evidence from functional studies, patient phenotyping, and population data. The absence of specific variant details in the query implies the variant in question is one with disease relevance, as CHD7 is a critical gene for development and mutations in this gene are consistent

## 22. Build the results dataframe (4b)
Applies the parser to get `predicted_label` for every row.

In [22]:
import pandas as pd

results_coding_4b_df = pd.DataFrame(lines_4b)
results_coding_4b_df['predicted_label'] = results_coding_4b_df.apply(
    lambda row: parse_prediction_v3(row['content'], row['reasoning']), axis=1
)

print(results_coding_4b_df['predicted_label'].value_counts())

predicted_label
Benign        128
Pathogenic     71
Uncertain       1
Name: count, dtype: int64


## 23. Evaluate qwen3:4b performance
Computes accuracy, precision, recall, and F1 (macro and per-class), for comparison against the 1.7b run.

In [23]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, f1_score, precision_score, recall_score

y_true_coding_4b = results_coding_4b_df['true_label']
y_pred_coding_4b = results_coding_4b_df['predicted_label']

accuracy_c4 = accuracy_score(y_true_coding_4b, y_pred_coding_4b)
precision_c4, recall_c4, f1_c4, _ = precision_recall_fscore_support(
    y_true_coding_4b, y_pred_coding_4b, labels=['Pathogenic', 'Benign'], average='macro', zero_division=0
)

f1_scores_c4 = f1_score(y_true_coding_4b, y_pred_coding_4b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
precision_scores_c4 = precision_score(y_true_coding_4b, y_pred_coding_4b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
recall_scores_c4 = recall_score(y_true_coding_4b, y_pred_coding_4b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)

print(f"Accuracy:  {accuracy_c4*100:.2f}%")
print(f"Macro Precision: {precision_c4*100:.2f}%")
print(f"Macro Recall:    {recall_c4*100:.2f}%")
print(f"Macro F1-Score:  {f1_c4*100:.2f}%")

print(f"\nPathogenic-class F1:        {f1_scores_c4[0]*100:.2f}%")
print(f"Pathogenic-class Precision: {precision_scores_c4[0]*100:.2f}%")
print(f"Pathogenic-class Recall:    {recall_scores_c4[0]*100:.2f}%")

print("\n--- Full classification report ---")
print(classification_report(y_true_coding_4b, y_pred_coding_4b, labels=['Pathogenic', 'Benign'], zero_division=0))

Accuracy:  77.50%
Macro Precision: 81.87%
Macro Recall:    79.24%
Macro F1-Score:  77.48%

Pathogenic-class F1:        75.14%
Pathogenic-class Precision: 95.77%
Pathogenic-class Recall:    61.82%

--- Full classification report ---
              precision    recall  f1-score   support

  Pathogenic       0.96      0.62      0.75       110
      Benign       0.68      0.97      0.80        90

   micro avg       0.78      0.78      0.78       200
   macro avg       0.82      0.79      0.77       200
weighted avg       0.83      0.78      0.77       200

